# Multilingual Sentiment Classification — Fine-tuned DistilBERT
This notebook fine-tunes **`distilbert-base-multilingual-cased`** on the course reviews dataset
to classify reviews as **negative / neutral / positive**.

**Before running:** In Colab, go to `Runtime` → `Change runtime type` → set **Hardware accelerator = GPU (T4)**.


## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn evaluate


## 2. Upload and load the dataset
Upload `reviews_by_course.csv` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select reviews_by_course.csv


In [ ]:
import pandas as pd

df = pd.read_csv('reviews_by_course.csv')
df = df.dropna(subset=['Review']).copy()
df['Review'] = df['Review'].astype(str)

def label_sentiment(r):
    if r <= 2:
        return 0  # negative
    elif r == 3:
        return 1  # neutral
    else:
        return 2  # positive

df['label'] = df['Label'].apply(label_sentiment)
label_names = ['negative', 'neutral', 'positive']
print(df['label'].value_counts())
df[['Review', 'label']].head()


## 3. Handle class imbalance
The dataset is heavily skewed toward positive reviews (~92%). We'll:
- Downsample the positive class a bit, and
- Use class weights in the loss function
so the model doesn't just predict "positive" for everything.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

train_df, test_df = train_test_split(
    df[['Review', 'label']], test_size=0.15, random_state=42, stratify=df['label']
)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_df['label'].values
)
print("Class weights (negative, neutral, positive):", class_weights)
print("Train size:", len(train_df), "Test size:", len(test_df))


## 4. Tokenize with the multilingual DistilBERT tokenizer

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df.rename(columns={'Review': 'text'}), preserve_index=False)
test_ds = Dataset.from_pandas(test_df.rename(columns={'Review': 'text'}), preserve_index=False)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column('label', 'labels')
test_ds = test_ds.rename_column('label', 'labels')
train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


## 5. Load model + custom weighted-loss Trainer (to handle class imbalance)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


## 6. Training arguments + metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1_macro': f1, 'precision_macro': precision, 'recall_macro': recall}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=50,
    report_to='none'
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)


## 7. Train (this needs the GPU runtime — will be slow on CPU)

In [ ]:
trainer.train()


## 8. Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

preds_output = trainer.predict(test_ds)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print(classification_report(y_true, y_pred, target_names=label_names, digits=3))
print(confusion_matrix(y_true, y_pred))


## 9. Save the fine-tuned model

In [ ]:
model.save_pretrained('./sentiment_distilbert_multilingual')
tokenizer.save_pretrained('./sentiment_distilbert_multilingual')

# zip and download
!zip -r sentiment_distilbert_multilingual.zip sentiment_distilbert_multilingual
from google.colab import files
files.download('sentiment_distilbert_multilingual.zip')


## 10. Try it on a new review

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=-1).item()
    return label_names[pred]

print(predict_sentiment("This course was amazing, I learned so much!"))
print(predict_sentiment("Ce cours était décevant et mal organisé."))  # French: disappointing
print(predict_sentiment("这门课程很好，内容很实用"))  # Chinese: good course
